In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# =====================================================================
# 1. Device and Seed Configuration
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

batch_size = 128
num_epochs = 15  # Standardized for fair comparison

# Base transformations for standardized evaluation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Loading the CIFAR-10 dataset
train_loader = DataLoader(datasets.CIFAR10(root='./data', train=True, download=True, transform=transform), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(datasets.CIFAR10(root='./data', train=False, download=True, transform=transform), batch_size=batch_size, shuffle=False)

# =====================================================================
# 2. Reference Models (From Your Previous Tasks)
# =====================================================================
class InitialMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class InitialCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.fc1 = nn.Linear(256 * 2 * 2, 256)
        self.fc2 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = F.max_pool2d(torch.sigmoid(self.conv1(x)), 2)
        x = F.max_pool2d(torch.sigmoid(self.conv2(x)), 2)
        x = F.max_pool2d(torch.sigmoid(self.conv3(x)), 2)
        x = F.max_pool2d(torch.sigmoid(self.conv4(x)), 2)
        x = x.view(x.size(0), -1)
        return self.fc2(torch.sigmoid(self.fc1(x)))

# =====================================================================
# 3. CREATIVITY TASK: Patch-Based MLP + CNN Fusion Network
# =====================================================================
class PatchBasedMlpCnnFusion(nn.Module):
    def __init__(self, patch_size=4, in_channels=3, embed_dim=64):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        
        # CIFAR-10 images are 32x32. Patch size 4x4 splits the image into (32/4)*(32/4) = 8x8 = 64 patches.
        # Flattened feature dimension of a single patch: 4 * 4 * 3 (channels) = 48.
        patch_flat_dim = patch_size * patch_size * in_channels
        
        # --- Stage 1: Patch-based MLP Encoder ---
        # Independently maps each local patch into a continuous embedding space
        self.mlp_stage1 = nn.Linear(patch_flat_dim, 128)
        self.mlp_stage2 = nn.Linear(128, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)
        
        # --- Stage 2: CNN Fusion Layers ---
        # Treats the reconstructed 8x8 grid of patch embeddings as a 2D topological map
        self.conv1 = nn.Conv2d(embed_dim, 128, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(128)
        self.conv2 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(256)
        
        # --- Stage 3: Global Classifier ---
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc_out = nn.Linear(256, 10)

    def forward(self, x):
        B, C, H, W = x.shape  # Input batch size, channels, height, width [B, 3, 32, 32]
        P = self.patch_size
        
        # 1. Tokenization: Extract non-overlapping patches and flatten them
        # Resulting shape after unfolding and reshaping: [B, 64 patches, 48 features]
        patches = x.unfold(2, P, P).unfold(3, P, P)
        patches = patches.permute(0, 2, 3, 4, 5, 1).contiguous()
        patches = patches.view(B, (H // P) * (W // P), -1)
        
        # 2. MLP Feature Projection: Process patches locally
        x_mlp = F.relu(self.mlp_stage1(patches))
        x_mlp = self.layer_norm(self.mlp_stage2(x_mlp))  # Shape: [B, 64, embed_dim]
        
        # 3. Spatial Reconstruction (Fusion): Reshape back to a 2D grid for spatial CNN processing
        grid_h, grid_w = H // P, W // P
        x_fusion = x_mlp.permute(0, 2, 1).contiguous().view(B, self.embed_dim, grid_h, grid_w)  # Shape: [B, embed_dim, 8, 8]
        
        # 4. CNN Hierarchical Aggregation: Convolve over structural coordinates of patches
        x_cnn = F.relu(self.bn1(self.conv1(x_fusion)))
        x_cnn = F.max_pool2d(x_cnn, 2)  # Downsamples grid size from 8x8 to 4x4
        
        x_cnn = F.relu(self.bn2(self.conv2(x_cnn)))
        x_cnn = self.global_pool(x_cnn)  # Compresses structural dimensions into a dense vector: [B, 256, 1, 1]
        
        # 5. Output Projection for 10-class logit generation
        x_cnn = x_cnn.view(B, -1)
        return self.fc_out(x_cnn)

# =====================================================================
# 4. Universal Pipeline Function
# =====================================================================
def run_pipeline(model_class, name):
    print(f"\n--- Training {name} ---")
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = {'train_loss': [], 'test_acc': []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                _, predicted = model(images).max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total
        history['train_loss'].append(epoch_loss)
        history['test_acc'].append(epoch_acc)
        print(f"Epoch {epoch+1:02d} | Loss: {epoch_loss:.4f} | Test Acc: {epoch_acc:.2f}%")

    return history

# =====================================================================
# 5. Execution and Plotting
# =====================================================================
results = {
    'Standard MLP': run_pipeline(InitialMLP, 'Standard MLP'),
    'Standard CNN (Sigmoid)': run_pipeline(InitialCNN, 'Standard CNN (Sigmoid)'),
    'Patch-MLP CNN Fusion': run_pipeline(PatchBasedMlpCnnFusion, 'Patch-MLP CNN Fusion')
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plotting Training Loss Convergence Curves
for name, metrics in results.items():
    axes[0].plot(range(1, num_epochs + 1), metrics['train_loss'], label=name, marker='o')
axes[0].set_title('Training Loss Convergence')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Plotting Test Accuracy Metric Comparison
for name, metrics in results.items():
    axes[1].plot(range(1, num_epochs + 1), metrics['test_acc'], label=name, marker='s')
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()